# Testscript for Master Thesis 

In [35]:
# Setup
# cd to path of current script
import os
import sys
import glob
os.chdir(sys.path[0])

In [9]:
# Download tweet data from Annotation Sensitivity Paper (kern, 2023)
from huggingface_hub import hf_hub_download
import pandas as pd

REPO_ID = "soda-lmu/tweet-annotation-sensitivity-2"
FILENAME = "publication_dataset.csv"

dataset = pd.read_csv(
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset")
)

publication_dataset.csv:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

In [25]:
# Get Original Davidon Labels as Gold Standard
df_unique = dataset.drop_duplicates(
    subset=['tweet_id', 'tweet_hashed', 'orig_label_hs', 'orig_label_ol', 'orig_label_ne']
).reset_index(drop=True)[['tweet_id', 'tweet_hashed', 'orig_label_hs', 'orig_label_ol', 'orig_label_ne']]
df_unique = df_unique.dropna(subset=['tweet_hashed'])
df_unique_sample = df_unique[0:10]
df_unique_sample

,tweet_id,tweet_hashed,orig_label_hs,orig_label_ol,orig_label_ne
0,1,@###### bro that hoe live,0.0,2.0,1.0
1,2,"Stop using the word ""redskin(s)"".",0.0,2.0,1.0
2,3,Now when I put this pussy on Vivian she bet no...,0.0,3.0,0.0
3,4,RT @###### Call her 1st thing in da AM.... Let...,0.0,3.0,0.0
4,5,I just want to punch you in the throat when yo...,2.0,1.0,0.0
5,6,@###### @###### fight me you fucking obese niglet,2.0,1.0,0.0
6,7,Really. 3 hours to find paper work is retarded...,0.0,2.0,1.0
7,8,' You Deserve x August Alsina &#1043358;&#1043...,0.0,2.0,1.0
8,9,@###### @###### trash,0.0,1.0,2.0
9,10,@###### I hate that nigger.,3.0,0.0,0.0


In [28]:
# Generate 10 Chat GPT Labels for each tweet in the sample data 
import openai
from api_key import API_KEY as API_KEY

In [56]:
from openai import OpenAI

client = OpenAI(
  api_key=API_KEY
)

def classify_tweet(tweet_text):
    """
    Sends a prompt to ChatGPT asking it to categorize the tweet as:
    - "HS" if it is hate speech
    - "OL" if it is offensive language
    - "neither" otherwise
    Returns a string: "HS", "OL", or "neither".
    """
    # Create a conversation with system & user messages
    response = client.chat.completions.create(
        # model="gpt-4o-mini",  # or 'gpt-4' if you have access
        model="gpt-3.5-turbo", 
        temperature=1,
        max_tokens=10,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant specialized in classifying tweets."
                    "You will have to classify these tweets as either containing:\n"
                    "A) Hate Speech (HS), B) Offensive Language but not Hate Speech (OL), or "
                    "C) neither Hate Speech or Offensive Language (neither)"
                    "Given a tweet, respond with ONLY one of these exact labels:\n\n"
                    "HS\nOL\nneither"
                )
            },
            {
                "role": "user",
                "content": f"Tweet: {tweet_text}\n\nIs it HS, OL, or neither?"
            },
        ],
    )

    # Extract the assistant's answer
    label = response.choices[0].message.content

    # In case GPT responds with text around the label, just take the first word
    # or do some basic cleaning. But ideally, it should match exactly "HS", "OL", or "neither".
    if label not in ["HS", "OL", "neither"]:
        label = label.split()[0]  # fallback, just in case there is extra text

    # Return the label
    return label



In [59]:
import time

def annotate_tweets(input_csv, output_csv, num_repetitions=5, sleep_seconds=1):
    """
    Reads tweets from input_csv (which has at least the column 'tweet_hashed'),
    asks ChatGPT `num_repetitions` times per tweet to label it as HS, OL, or neither,
    aggregates the counts, and writes the result to output_csv.
    """
    # Read the CSV
    df = df_unique_sample

    # Prepare columns for counts (initialize them with 0)
    df.loc[:, "HS_count"] = 0
    df.loc[:, "OL_count"] = 0
    df.loc[:, "neither_count"] = 0

    # Loop over each tweet
    for i, row in df.iterrows():
        tweet_text = row["tweet_hashed"]

        hs_cnt = 0
        ol_cnt = 0
        neither_cnt = 0

        # Ask ChatGPT multiple times for each tweet
        for _ in range(num_repetitions):
            label = classify_tweet(tweet_text)

            if label == "HS":
                hs_cnt += 1
            elif label == "OL":
                ol_cnt += 1
            else:
                neither_cnt += 1

            # Sleep to avoid hitting rate limits too quickly
            time.sleep(sleep_seconds)

        # Store aggregated counts back into the dataframe
        df.loc[i, "HS_count"] = hs_cnt
        df.loc[i, "OL_count"] = ol_cnt
        df.loc[i, "neither_count"] = neither_cnt

    # Write out the updated DataFrame
    df.to_csv(output_csv, index=False)
    print(f"Annotated tweets saved to {output_csv}")

# Example usage:
if __name__ == "__main__":
    # Adjust paths, number of repetitions, etc. as needed
    annotate_tweets(
        input_csv="tweets.csv", 
        output_csv="tweets_annotated.csv", 
        num_repetitions=1, # adjust  
        sleep_seconds=22
    )

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [60]:
import pandas as pd
df_annotated =  pd.read_csv("tweets_annotated.csv")
df_annotated.head()

,tweet_id,tweet_hashed,orig_label_hs,orig_label_ol,orig_label_ne,HS_count,OL_count,neither_count
0,1,@###### bro that hoe live,0.0,2.0,1.0,0,3,0
1,2,"Stop using the word ""redskin(s)"".",0.0,2.0,1.0,0,0,3
2,3,Now when I put this pussy on Vivian she bet no...,0.0,3.0,0.0,0,1,2
3,4,RT @###### Call her 1st thing in da AM.... Let...,0.0,3.0,0.0,0,3,0
4,5,I just want to punch you in the throat when yo...,2.0,1.0,0.0,0,3,0


In [11]:
# Default Prompts and tasks 
# Text Only Tasks
return_number = "Return me a number between 0 and 50."
detect_OS0 = """Please tell me whether the following tweet does contain offensive language. 
               If so, only return 'OS' if not return 'non-OS' 
               Here is the tweet: 
               I live for two-bite brownies"""
detect_OS1 = """Please tell me whether the following tweet does contain offensive language. 
               If so, only return 'OS' if not return 'non-OS' 
               Here is the tweet: 
               @###### see bitch this is why you have no friends"""



In [12]:
# base Request
# test capabilties
from openai import OpenAI

client = OpenAI(
  api_key=API_KEY
)

completion = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", 
     # "content": return_number
     # "content": detect_OS0
     "content": rate_image
     # "content": rate_image2
    }
  ],
  temperature = 1 # default temp = 1  
)

print("Response: ", completion.choices[0].message.content);

RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o-mini in organization org-UaAY97exgMtNAHsPKi6KAm83 on tokens per min (TPM): Limit 60000, Requested 211287. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

### Accessing the logprobs?

In [2]:
import numpy as np
from IPython.display import display, HTML

def get_completion(
    messages: list[dict[str, str]],
    model: str = "gpt-4",
    max_tokens=500,
    temperature=0,
    stop=None,
    seed=123,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.chat.completions.create(**params)
    return completion

In [178]:
CLASSIFICATION_PROMPT = """You will be given a headline of a news article.
Please tell me whether the following tweet does contain offensive language. 
Please return me only the following: 
1) Return 'OS' if the tweet does contains Offensive language (if not return 'non-OS')
2) Return a a number from 1 to 4 representing your certainty rating for the decision (1 is low certainty, 4 is high certainty). 
Here is the tweet: {tweet}"""

In [179]:
tweets = [
    "I live for two-bite brownies",
    "@###### see bitch this is why you have no friends",
    "I f*****g love this"
]

In [189]:
API_RESPONSE.choices[0].message.content

'OS, 4'

In [190]:
for tweet in tweets:
    print(f"\nTweet: {tweet}")
    API_RESPONSE = get_completion(
        [{"role": "user", "content": CLASSIFICATION_PROMPT.format(tweet=tweet)}],
        model="gpt-4o-mini",
        logprobs=True,
        top_logprobs=5,
        temperature=1
    )
    
    # Extract and print the response text
    response_text = API_RESPONSE.choices[0].message.content
    print("Response: " + response_text)
    
    top_two_logprobs = API_RESPONSE.choices[0].logprobs.content[0].top_logprobs
    html_content = ""
    for i, logprob in enumerate(top_two_logprobs, start=1):
        html_content += (
            f"<span style='color: cyan'>Output token {i}:</span> {logprob.token}, "
            f"<span style='color: darkorange'>logprobs:</span> {logprob.logprob}, "
            f"<span style='color: magenta'>linear probability:</span> {np.round(np.exp(logprob.logprob)*100,2)}%<br>"
        )
    display(HTML(html_content))
    print("\n")


Tweet: I live for two-bite brownies
Response: non-OS, 4





Tweet: @###### see bitch this is why you have no friends
Response: OS, 4





Tweet: I f*****g love this
Response: OS, 4
